In [1]:
import os
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer
import shap

/Users/antoninbenard/PycharmProjects/PythonProject3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(os.getcwd())
os.chdir("../")
os.getcwd()

/Users/antoninbenard/PycharmProjects/PythonProject3


'/Users/antoninbenard/PycharmProjects'

In [3]:
df_test = pd.read_parquet("cours_sas/data/idf_vf_test_cleaned.parquet")
df_train = pd.read_parquet("cours_sas/data/idf_vf_train_cleaned.parquet")

In [4]:
df_full = pd.concat([df_train, df_test])

In [5]:
cols_to_remove = ['prix_m2', 'ecart_prix_median_pct', 'is_maison', 'nord_paris', 'est_paris']
X = df_full.drop(columns=cols_to_remove)
y = df_full['prix_m2']
print(f"\n Dataset: {X.shape[0]:,} échantillons, {X.shape[1]} features")
print(f"Target (prix_m2): min={y.min():.0f}, max={y.max():.0f}, médiane={y.median():.0f}")


 Dataset: 634,236 échantillons, 103 features
Target (prix_m2): min=1500, max=15000, médiane=4762


In [7]:
# Créer 10 bins pour stratification
n_bins = 10
binner = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile')
y_binned = binner.fit_transform(y.values.reshape(-1, 1)).ravel().astype(int)

# Vérifier la distribution
print(f"\n Target divisée en {n_bins} bins (quantiles)")
bin_counts = pd.Series(y_binned).value_counts().sort_index()
print("\nDistribution des bins:")
for bin_idx, count in bin_counts.items():
    print(f"   Bin {int(bin_idx)}: {count:>6} échantillons ({count/len(y)*100:>5.1f}%)")


 Target divisée en 10 bins (quantiles)

Distribution des bins:
   Bin 0:  64429 échantillons ( 10.2%)
   Bin 1:  63609 échantillons ( 10.0%)
   Bin 2:  63409 échantillons ( 10.0%)
   Bin 3:  63008 échantillons (  9.9%)
   Bin 4:  62982 échantillons (  9.9%)
   Bin 5:  62689 échantillons (  9.9%)
   Bin 6:  63386 échantillons ( 10.0%)
   Bin 7:  63229 échantillons ( 10.0%)
   Bin 8:  63532 échantillons ( 10.0%)
   Bin 9:  63963 échantillons ( 10.1%)


/Users/antoninbenard/PycharmProjects/PythonProject3/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


In [8]:
n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

shap_results = []
correlation_results = []


for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y_binned), 1):
    print(f" FOLD {fold_idx}/{n_folds}")

    # Split stratifié
    X_fold = X.iloc[train_idx]
    y_fold = y.iloc[train_idx]

    print(f"   Train: {len(train_idx):,} échantillons")
    print(f"   Val:   {len(val_idx):,} échantillons")

    # Vérifier la distribution de la target dans le fold
    y_fold_stats = {
        'min': y_fold.min(),
        'q25': y_fold.quantile(0.25),
        'median': y_fold.median(),
        'q75': y_fold.quantile(0.75),
        'max': y_fold.max()
    }
    print(f"   Target train: min={y_fold_stats['min']:.0f}, "
          f"Q25={y_fold_stats['q25']:.0f}, "
          f"med={y_fold_stats['median']:.0f}, "
          f"Q75={y_fold_stats['q75']:.0f}, "
          f"max={y_fold_stats['max']:.0f}")


    lgbm = LGBMRegressor(
        n_estimators=150,
        max_depth=8,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42 + fold_idx,
        verbose=-1,
        n_jobs=-1
    )
    lgbm.fit(X_fold, y_fold)


    # Prendre un échantillon représentatif (stratifié)
    sample_size = min(1000, len(X_fold))
    sample_indices = []

    # Échantillonner de manière stratifiée
    y_fold_binned = binner.transform(y_fold.values.reshape(-1, 1)).ravel().astype(int)
    for bin_val in range(n_bins):
        bin_indices = np.where(y_fold_binned == bin_val)[0]
        n_samples_bin = int(sample_size / n_bins)
        if len(bin_indices) > 0:
            sampled = np.random.choice(bin_indices,
                                      size=min(n_samples_bin, len(bin_indices)),
                                      replace=False)
            sample_indices.extend(sampled)

    X_sample = X_fold.iloc[sample_indices]

    explainer = shap.TreeExplainer(lgbm)
    shap_values = explainer.shap_values(X_sample)

    # Importance moyenne absolue
    mean_abs_shap = np.abs(shap_values).mean(axis=0)

    shap_results.append({
        'fold': fold_idx,
        'shap_values': mean_abs_shap,
        'feature_names': X.columns.tolist()
    })

 FOLD 1/5
   Train: 507,388 échantillons
   Val:   126,848 échantillons
   Target train: min=1500, Q25=3308, med=4762, Q75=8000, max=15000
 FOLD 2/5
   Train: 507,389 échantillons
   Val:   126,847 échantillons
   Target train: min=1500, Q25=3308, med=4762, Q75=8000, max=15000
 FOLD 3/5
   Train: 507,389 échantillons
   Val:   126,847 échantillons
   Target train: min=1500, Q25=3308, med=4762, Q75=7998, max=15000
 FOLD 4/5
   Train: 507,389 échantillons
   Val:   126,847 échantillons
   Target train: min=1500, Q25=3308, med=4762, Q75=8000, max=15000
 FOLD 5/5
   Train: 507,389 échantillons
   Val:   126,847 échantillons
   Target train: min=1500, Q25=3308, med=4762, Q75=8000, max=15000


In [9]:
shap_matrix = np.array([result['shap_values'] for result in shap_results])
shap_mean = shap_matrix.mean(axis=0)
shap_std = shap_matrix.std(axis=0)


results_df = pd.DataFrame({
    'feature': X.columns,
    'shap_mean': shap_mean,
    'shap_std': shap_std
})

# Normaliser les scores (0-100)
results_df['shap_norm'] = (results_df['shap_mean'] - results_df['shap_mean'].min()) / \
                          (results_df['shap_mean'].max() - results_df['shap_mean'].min()) * 100




# Trier par score combiné
results_df = results_df.sort_values('shap_norm', ascending=False)

results_df[['feature', 'shap_mean', 'shap_norm']]

,feature,shap_mean,shap_norm
3,prix_m2_median_maisons_voisines,1328.617232,100.000000
92,ratio_terrain_bati,309.631535,23.301283
25,commune_education_score_2013_commune,231.257324,17.402082
101,surface_x_type,223.225635,16.797540
90,densite_rel_2000m,216.864588,16.318746
...,...,...,...
50,nb_restauration_500m,0.120492,0.004489
52,nb_sante_500m,0.119039,0.004380
48,nb_commerce_500m,0.102690,0.003149
62,nb_transport_autre_500m,0.068553,0.000580


In [10]:
selected_features = results_df[results_df["shap_norm"] > 1]["feature"].tolist()
df_full = df_full[selected_features]
df_full
selected_features

['prix_m2_median_maisons_voisines',
 'ratio_terrain_bati',
 'commune_education_score_2013_commune',
 'surface_x_type',
 'densite_rel_2000m',
 'nombre_lots',
 'nb_transport_autre_2000m',
 'commune_part_commerce_tourisme',
 'is_appartement',
 'log_surface_bati',
 'distance_paris',
 'commune_densite_residences_principales_2020',
 'densite_lots',
 'commune_revenu_median_2020',
 'nb_jours_depuis_janvier_2021',
 'commune_etablissements_1_salarie_2021',
 'Taux_moyen_credit_immo_lag3t',
 'commune_ratio_residences_secondaires_population',
 'prix_median_x_distance',
 'commune_idh2_2013_commune',
 'surface_par_piece',
 'dist_transport_lourd',
 'surface_terrain',
 'Taux_moyen_credit_immo_lag1t',
 'nb_loisirs_2000m',
 'Evolution_PIB_volume_en_%_base2020_lag4t',
 'commune_taux_proprietaires',
 'nombre_pieces_principales',
 'nb_services_2000m',
 'nb_education_2000m',
 'dist_transport_autre']